In [ ]:
ah = 10.0
bandwidth = 1
diameter = 2.0
frequency = 1000.0
antenna_efficiency = 0.6
pulse_width = 1

wavelength = 0.299792458
transmitter_antenna_gain = 24.20869581244019
receiver_antenna_gain = 24.20869581244019
target_rcs = 2.0

power = 20_000
cpi_pulses = 1
noise_temperature = 300
rf_loss = 12.0
pfa = 1e-6

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
from scipy import integrate, special
import scipy.constants as sc


def marcum_q_integrand(v: float, alpha: float) -> float:
    return v * np.exp(-(v * v + alpha * alpha) / 2) * special.iv(0, alpha * v)


def marcum_q_function(alpha: float, beta: float) -> float:
    """
    Calculate the values of the marcum q function.

    Parameters
    ----------
    alpha: float
        alpha parameter; must be <= 30
    beta: float
        beta parameter

    Returns
    -------
    float

    Raises
    ------
    ValueError
        If alpha is > 30 due to numerical instability of the Bessel function
    """
    if alpha > 30.0:
        raise ValueError("alpha must be <= 30 to avoid overflow in Bessel functions!")
    return (
        1
        - integrate.quad(
            lambda x: marcum_q_integrand(x, alpha),
            0,
            beta,
        )[0]
    )


def calculate_snr(
    wavelength: float,
    antenna_gain_transmitter: float,
    antenna_gain_receiver: float,
    radar_cross_section: float,
    distance_transmitter_target: float,
    distance_receiver_target: float,
    transmission_power: float,
    bandwidth: float,
    cpi_pulses: int,
    equivalent_temperature: float,
    L_t: float,
    L_a: float,
    polarization_factor: float,
    pattern_propagation_factor_transmitter: float,
    pattern_propagation_factor_receiver: float,
) -> float:
    r"""
    Calculate the signal-to-noise ratio (SNR) [dB] for free propagation.

    Parameters
    ----------
    wavelength: float
        Signal wavelength [m]
    antenna_gain_transmitter: float
        Antenna gain of the transmitter [dBi]
    antenna_gain_receiver: float
        Antenna gain of the receiver [dBi]
    radar_cross_section: float
        Radar cross section of the target [m^2]
    distance_transmitter_target: float
        Line-of-sight distance between the transmitter and the target [m]
    distance_receiver_target: float
        Line-of-sight distance between the receiver and the target [m]
    transmission_power: float
        Power of the signal [W]
    bandwidth: float
        Receiver bandwidth of the signal [MHz]
    cpi_pulses:
        Number of pulses within a Coherent Processing Interval (CPI)
    equivalent_temperature: float
        Equivalent temperature [K]
    L_t: float
        Transmission line loss [dB]
    L_a: float
        Atmospheric and precipitation attenuation [dB]
    polarization_factor: float
        Polarization factor [dB]
    pattern_propagation_factor_transmitter: float
        Pattern propagation factor for the path from the transmitter
        to the target [dB]
    pattern_propagation_factor_receiver: float
        Pattern propagation factor for the path from the target
        to the receiver [dB]

    Returns
    -------
    snr: float
        Signal-to-noise ratio

    Assumptions
    -----------
    - There is a direct line-of-sight.
    - No propagation losses.
    - No terrain losses.

    Notes
    -----
    Details about the implemented formula can be found in the docs at :ref:`snr-section`.
    The pulse width was replaced by the corresponding noise bandwidth
    :math:`B_n \approx \frac{1}{\tau}`.
    """
    # Scale to dB units.
    power_dB = 10 * np.log10(transmission_power)
    coherent_integration_gain_dB = 10 * np.log10(cpi_pulses)
    lambda_sq_dB = 2 * 10 * np.log10(wavelength)
    rcs_dB = 10 * np.log10(radar_cross_section)
    four_pi_dB = 10 * np.log10(pow((4 * np.pi), 3))
    ktb = 10 * np.log10(sc.Boltzmann * equivalent_temperature)

    bw_dB = 10 * np.log10(bandwidth * 1e6)

    snr = (
        power_dB
        + coherent_integration_gain_dB
        + antenna_gain_transmitter
        + antenna_gain_receiver
        + lambda_sq_dB
        + rcs_dB
        + 2 * polarization_factor
        + pattern_propagation_factor_transmitter
        + pattern_propagation_factor_receiver
        - four_pi_dB
        - ktb
        - bw_dB
        - 20 * np.log10(distance_transmitter_target)
        - 20 * np.log10(distance_receiver_target)
        - L_t
        - L_a
    )

    return snr


def calculate_probability_of_detection(snr: float, pfa: float) -> float:
    """
    Calculate the probability of detection.

    Parameters
    ----------
    snr: float
        Signal-to-noise ratio [dB]
    pfa: float
        probability of false alarm in (0, 1]

    Notes
    -----
    This function implements Equ. (22), (23) in Brennan & Reed, 1973.

    References
    ----------
    Brennan, L. E., & Reed, L. S. (1973). Theory of Adaptive Radar.
    IEEE Transactions on Aerospace and Electronic Systems, AES-9(2), 237-252.
    https://doi.org/10.1109/taes.1973.309792
    """
    alpha = pow(10, ((snr + 3) / 20))
    # Avoid blowup in Bessel function.
    if alpha > 30:
        return 1.0
    beta = math.sqrt(-2 * (np.log(pfa)))
    return marcum_q_function(alpha, beta)


atmospheric_loss_per_distance = 1.1e-05

rs = np.linspace(1e-6, 20_000, 20_001)

snrs = []
pds = []
for r in rs:
    snr = calculate_snr(
        wavelength=wavelength,
        antenna_gain_transmitter=transmitter_antenna_gain,
        antenna_gain_receiver=receiver_antenna_gain,
        radar_cross_section=target_rcs,
        distance_transmitter_target=r,
        distance_receiver_target=r,
        transmission_power=power,
        bandwidth=bandwidth,
        cpi_pulses=cpi_pulses,
        equivalent_temperature=noise_temperature,
        L_t=rf_loss,
        L_a=atmospheric_loss_per_distance * r,
        # TODO: Should we include these factors?
        polarization_factor=0.0,
        pattern_propagation_factor_receiver=0.0,
        pattern_propagation_factor_transmitter=0.0,
    )
    p = calculate_probability_of_detection(snr, pfa)

    snrs.append(snr)
    pds.append(p)


fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(2 * 8, 4.5))

ax = axes[0]
ax.plot(rs, snrs)
ax.set_xlabel("One-way Distance radar - target [m]")
ax.set_ylabel("SNR [dB]")
ax.grid(True)

ax = axes[1]
ax.plot(rs, np.array(pds) * 100)
ax.set_xlabel("One-way Distance radar - target [m]")
ax.set_ylabel("Probability of detection [%]")
ax.grid(True)
ax.axvline(17838, 0, 0.8, color="red")
ax.set_ylim(0, 100)

In [ ]:
import math

import numpy as np

ah = 10.0
bandwidth = 1
diameter = 2.0
frequency = 1000.0
antenna_efficiency = 0.6
pulse_width = 1

wavelength = 0.299792458
transmitter_antenna_gain = 24.20869581244019
receiver_antenna_gain = 24.20869581244019
target_rcs = 2.0

power = 20_000
cpi_pulses = 1
noise_temperature = 300
rf_loss = 12.0
pfa = 1e-6


def marcum_q_fn(v):
    """integrand function to evaluate the Marcum Q
    function that provides the probability of detection for
    a single pulse out of a quadrature detector.
    """
    return v * math.exp(-(v * v + alpha * alpha) / 2) * special.iv(0, alpha * v)


def radar_eq_max_dist(
    trans_pwr, antenna_diam, frequency, pulse_width, cpi_pulses, bandwidth, pfa, rcsSM
):
    """! returns maximal distance given the radar parameters and the target rcs.
    MAX RANGE IS SET TO 400kms, due to the HARD LIMIT in SPLAT! (see MAXPAFES in splatBurst.h)
    """

    ####################### Radar and target values (working) ##################

    c = sc.speed_of_light
    # speed of light
    rf_loss = 12
    # RF system hardware loss (not known for ASR, best guess from chapter 2.12, p.80 Skolnik)
    rcs_start = 10 * np.log10(rcsSM)
    # rcs_start=13;                        # RCS in db
    noise_figure = 1.9
    # Receiver LNA noise figure in dB  (not known for TA, best guess)
    window = 0
    # rectangular for no window, or Hamming (=0)
    equiv_temp = 300
    # equivalent temperature [K] (not known for ASR, best guess)

    ########################################################################################

    ############# ------------- analyze for upto 400 kms
    start_range = 0.001
    # [m] starting at 0 will cause a division by 0 error further down
    # Attention!!!!!! if the snr is very high (that is for low ranges) the besseli function will overflow
    # throwing a segmentation fault (another way to avoid it is to return Pd=1 for snr > e.g. 30dB)
    # this can be avoided by setting the start_range to be a higher value
    stop_range = 400000
    # [m]

    # ------------------------------evaluate  range resolution ---------------

    res = c / (2 * (bandwidth * 1e6))
    if window == 0:
        res = res * 1.44

    # we set resolution manually
    res = 1000  # [m]

    # -----------------------evaluate radar range equation -----------------------------------

    A = np.pi * antenna_diam * antenna_diam / 4
    wavelength = c / (frequency * 1e9)
    antenna_gain = 10 * np.log10(0.6 * 4 * np.pi * A / (wavelength * wavelength))

    four_pi = 10 * np.log10(pow((4 * np.pi), 3))
    pt = 10 * np.log10(trans_pwr)
    lambda_sq = 2 * 10 * np.log10(c / (frequency * 1e9))
    ktb = 10 * np.log10(1.38e-23 * equiv_temp * (bandwidth * 1e6))
    t_bw_gain = 10 * np.log10(pulse_width * bandwidth)
    dop_gain = 10 * np.log10(cpi_pulses)

    ranges = np.arange(start_range, stop_range + 1, res)
    snr = []
    for rng in ranges:
        curr_snr = (
            pt
            + lambda_sq
            + 2 * antenna_gain
            + t_bw_gain
            + dop_gain
            + rcs_start
            - four_pi
            - ktb
            - 40 * np.log10(rng)
            - noise_figure
            - rf_loss
        )
        snr = np.concatenate([snr, [curr_snr]])

    # -----------------------------evaluate Pd function -----------------------

    beta = math.sqrt(-2 * (np.log(pfa)))

    rangel = np.arange(start_range, stop_range + 1, res)

    # lrl = rangel.shape[0]

    jj = 0
    pd = []

    for rng in rangel:
        # avoid segmentation fault in the besseli function for high snr values
        if snr[jj] > 30:
            pd = np.concatenate([pd, [1.0]])
        else:
            global alpha  # make alpha global to pass it to marcum_r_fn
            alpha = pow(10, (snr[jj] + 3) / 20)
            # this is declared global above
            curr_pd = 1 - integrate.quad(marcum_q_fn, 0, beta)[0]
            pd = np.concatenate([pd, [curr_pd]])

        jj = jj + 1

    # compute and return max range (attention: all max ranges above this will not be noted by the user)
    retval = 400000

    ii = 0
    for _ in pd:
        if pd[ii] <= 0.8:
            retval = (rangel[ii] + rangel[ii - 1]) / 2
            break
        ii = ii + 1

    # change from m to km
    retval = retval / 1000.0
    return retval


radar_eq_max_dist(
    power,
    diameter,
    frequency / 1000,
    pulse_width,
    cpi_pulses,
    bandwidth,
    pfa,
    target_rcs,
)

In [ ]:
from theia.openburst_client import OpenburstClient


client = OpenburstClient("localhost", debugging=True)

In [ ]:
radar.receiver.noise_figure

In [ ]:
client.calculate_coverage(radar, 0.0, target_rcs)

In [ ]:
from theia.types import (
    MonostaticRadarMeasurementModel,
    MonostaticSensor,
    Point,
    Polarization,
    Receiver,
    Transmitter,
    calculate_antenna_gain,
)
from theia.util import frequency_to_wavelength


p = Point(
    lat=47.36700085728634,
    lon=8.537724304199216,
    alt=407.83600886023686,
)
ah = 10.0
bandwidth = 1
diameter = 2.0
frequency = 1000.0
antenna_efficiency = 0.6
radar = MonostaticSensor(
    id=0,
    transmitter=Transmitter(
        id=585,
        point=p,
        power=20000,
        erp=800,
        antenna_height=ah,
        antenna_diameter=diameter,
        antenna_gain=calculate_antenna_gain(
            diameter,
            frequency_to_wavelength(frequency),
            antenna_efficiency,
        ),
        frequency=frequency,
        pulse_width=1,
        bandwidth=bandwidth,
        polarization=Polarization.HORIZONTAL,
    ),
    receiver=Receiver(
        id=585,
        point=p,
        antenna_height=ah,
        diameter=2.0,
        cpi_pulses=1,
        pfa=1e-6,
        min_elevation=-20.0,
        max_elevation=60.0,
        rotation_time=10.0,
        bandwidth=bandwidth,
    ),
    error_model=MonostaticRadarMeasurementModel(),
)

In [ ]:
client.

In [ ]:
from theia.simulation.logging import LogLoader


loader = LogLoader("output.json")

In [ ]:
import itertools


pcl_detections = sorted(loader.blue_pcl_detections, key=lambda d: d.time)
for time, detections in itertools.groupby(pcl_detections, key=lambda d: d.time):
    detections = list(detections)
    if len(detections) >= 3:
        break

In [ ]:
import datetime


assert all([detection.target == detections[0].target for detection in detections])
detection = detections[0]
target = detection.target

t_next = detection.time + datetime.timedelta(seconds=1)
detections_next = [d for d in loader.blue_pcl_detections if d.time == t_next]
target_next = detections_next[0].target

In [ ]:
import numpy as np


np.array(target_next.point.as_tuple()) - np.array(target.point.as_tuple())

In [ ]:
from theia.mapping import pcl_detection_to_polygon


polygons = {
    f"Detection ID = {detection.detection_id}": pcl_detection_to_polygon(
        detection,
        target.alt,
        n_theta=360,
        n_phi=360,
    )
    for detection in detections
}
polygons2 = {
    f"Detection ID = {detection.detection_id}": pcl_detection_to_polygon(
        detection,
        target_next.alt,
        n_theta=360,
        n_phi=360,
    )
    for detection in detections_next
}
polygons.update(polygons2)

In [ ]:
from theia.mapping import RadarMap


map = RadarMap(
    sensors={f"ID {sensor.id}": sensor for sensor in loader.blue_pcl_sensors},
    targets={"target": target, "target_next": target_next},
    polygons=polygons,
).to_map()
map.location = (detection.sensor.receiver.lat, detection.sensor.receiver.lon)
map

In [ ]:
from theia.detection.pcl import PclDetector
from theia.test_data import load_pcl_example

sensors, trajcetories, grid = load_pcl_example()
trajectory = trajcetories[0]
RCS = trajectory.cross_section_model.rcs

In [ ]:
grid.model_dump_json()

In [ ]:
from typing import Any, Literal

import pydantic
import shapely

import theia
from theia.detection.pcl import pcl_track_init_update_masks
from theia.grids import LatLonHeightGrid
from theia.types import Sensor
from theia.util import mask_to_polygon


class GeoJSONPolygon(pydantic.BaseModel):
    type: Literal["Polygon"] = "Polygon"
    coordinates: list[list[list[float]]]

    @classmethod
    def from_shapely(cls, polygon: shapely.Polygon) -> "GeoJSONPolygon":
        geojson = shapely.geometry.mapping(polygon)
        return cls(
            coordinates=[list(map(list, ring)) for ring in geojson["coordinates"]]
        )


class GeoJSONMultiPolygon(pydantic.BaseModel):
    type: Literal["MultiPolygon"] = "MultiPolygon"
    # One extra nesting level: [polygon][ring][point][coordinate]
    coordinates: list[list[list[list[float]]]]

    @classmethod
    def from_shapely(cls, multi: shapely.MultiPolygon) -> "GeoJSONMultiPolygon":
        geojson = shapely.geometry.mapping(multi)
        return cls(
            coordinates=[
                [list(map(list, ring)) for ring in polygon]
                for polygon in geojson["coordinates"]
            ]
        )


GeoJSONGeometry = GeoJSONPolygon | GeoJSONMultiPolygon


class GeoJSONFeature(pydantic.BaseModel):
    type: str = "Feature"
    geometry: GeoJSONGeometry = pydantic.Field(discriminator="type")
    properties: dict[str, Any] = {}

    @classmethod
    def from_shapely(
        cls,
        shape: shapely.Polygon | shapely.MultiPolygon,
        properties: dict[str, Any] = {},
    ) -> "GeoJSONFeature":
        if isinstance(shape, shapely.Polygon):
            geometry = GeoJSONPolygon.from_shapely(shape)
        elif isinstance(shape, shapely.MultiPolygon):
            geometry = GeoJSONMultiPolygon.from_shapely(shape)
        else:
            raise TypeError(f"Unsupported geometry type: {type(shape)}")
        return cls(geometry=geometry, properties=properties)


def calculate_pcl_coverage(
    sensors: list[Sensor],
    grid: LatLonHeightGrid,
    rcs: float,
    snr_threshold: float = theia.config.SNR_THRESHOLD_PCL,
    doppler_threshold: float = theia.config.DOPPLER_SHIFT_THRESHOLD_PCL,
    delay_threshold: float = theia.config.DELAY_THRESHOLD_PCL,
) -> tuple[GeoJSONFeature, GeoJSONFeature]:
    """
    Calculate PCL coverage.

    Parameters
    ----------
    sensor: Sensor
        Sensor
    grid: LatLonHeightGrid
        Calculation grid
    rcs: float
        Radar cross section for which to calculate the coverage
    snr_threshold: float, default theia.config.SNR_THRESHOLD_PCL
        Minimum detectable threshold [dB]
    doppler_threshold: float, default theia.config.DOPPLER_SHIFT_THRESHOLD_PCL
        Minimum detectable Doppler shift [Hz]
    delay_threshold: float, default theia.config.DELAY_THRESHOLD_PCL
        Delay threshold for PCL [us].
        This is used to judge whether a given transmitter - target - receiver geometry
        is in the bistatic or the forward scattering regime.

    Returns
    -------
    track_init_coverage: GeoJSONFeature
        Region in which a track init can happen only using PCL
    track_update_coverage: GeoJSONFeature
        Region in which a track update can happen only using PCL
    """
    assert grid.altitude_values.shape[0] == 1

    detector = PclDetector(
        snr_threshold=snr_threshold,
        doppler_threshold=doppler_threshold,
        delay_threshold=delay_threshold,
    )

    track_init_mask, track_update_mask = pcl_track_init_update_masks(
        detector,
        sensors,
        grid,
        rcs,
    )

    polygons_init = mask_to_polygon(
        track_init_mask[:, :, 0],
        grid.latitude_values[0],
        grid.latitude_values[1] - grid.latitude_values[0],
        grid.longitude_values[0],
        grid.longitude_values[1] - grid.longitude_values[0],
    )
    import json

    # with open("polygons.json", "w") as file:
    #     json.dump(polygons_init, file)

    polygons_update = mask_to_polygon(
        track_update_mask[:, :, 0],
        grid.latitude_values[0],
        grid.latitude_values[1] - grid.latitude_values[0],
        grid.longitude_values[0],
        grid.longitude_values[1] - grid.longitude_values[0],
    )
    return (
        GeoJSONFeature(
            geometry=GeoJSONMultiPolygon.from_shapely(
                shapely.MultiPolygon(polygons_init)
            )
        ),
        GeoJSONFeature(
            geometry=GeoJSONMultiPolygon.from_shapely(
                shapely.MultiPolygon(polygons_update)
            )
        ),
    )


track_init, track_update = calculate_pcl_coverage([sensors[0]], grid, 1.0)

In [ ]:
from pydantic import TypeAdapter
from theia.types import Sensor

TypeAdapter(list[Sensor]).dump_json(sensors)

In [ ]:
sensors[0].model_dump_json()

In [ ]:
import cProfile

from theia.detection.pcl import pcl_track_init_update_masks

detector = PclDetector()

profiler = cProfile.Profile()
profiler.enable()

track_init_mask, track_update_mask = pcl_track_init_update_masks(
    detector,
    sensors,
    grid,
    RCS,
)

profiler.disable()
profiler.dump_stats("profile__pcl_coverage.prof")

In [ ]:
track_init_mask.shape

In [ ]:
grid.model_dump_json()

In [ ]:
import folium

from theia.mapping import RadarMap
from theia.util import mask_to_polygon


map = RadarMap(
    sensors={f"Sensor {sensor.id}": sensor for sensor in sensors},
    trajectories={"Target": trajectory},
).to_map()
map.location = (sensors[0].receiver.lat, sensors[0].receiver.lon)

polygons = mask_to_polygon(
    track_init_mask[:, :, 0],
    grid.latitude_values[0],
    grid.latitude_values[1] - grid.latitude_values[0],
    grid.longitude_values[0],
    grid.longitude_values[1] - grid.longitude_values[0],
)
for polygon in polygons:
    folium.GeoJson(polygon, fillColor="red", color="red").add_to(map)

polygons = mask_to_polygon(
    track_update_mask[:, :, 0],
    grid.latitude_values[0],
    grid.latitude_values[1] - grid.latitude_values[0],
    grid.longitude_values[0],
    grid.longitude_values[1] - grid.longitude_values[0],
)
for polygon in polygons:
    folium.GeoJson(polygon, fillColor="blue", color="blue").add_to(map)
map

In [ ]:
import numpy as np

from theia.detection.pcl import PclDetector
from theia.export_paraview import ParaviewExporter, PointOfInterest


pois: list[PointOfInterest] = []
poi_id = 0
for sensor in pcl_sensors:
    pois.append(
        PointOfInterest(
            id=poi_id,
            label=f"Rx {sensor.receiver.id}",
            type="Rx",
            lat=sensor.receiver.lat,
            lon=sensor.receiver.lon,
            alt=sensor.receiver.alt,
        )
    )
    poi_id += 1
    pois.append(
        PointOfInterest(
            id=poi_id,
            label=f"Tx {sensor.transmitter.id}",
            type="Tx",
            lat=sensor.transmitter.lat,
            lon=sensor.transmitter.lon,
            alt=sensor.transmitter.alt,
        )
    )
    poi_id += 1

exporter = ParaviewExporter(
    lat_min,
    lat_max,
    lat_res,
    lon_min,
    lon_max,
    lon_res,
)

sensor = pcl_sensors[0]

detector = PclDetector()

exporter.export_terrain(
    "test",
    # {
    #     "min detectable RCS": lambda *p: detector.minimum_detectable_rcs_vector(
    #         sensor.receiver, sensor.transmitter, np.array(p).reshape((1, 3))
    #     )
    # },
)
exporter.export_pois(pois, "pois.csv")